# Hybrid Retriever 성능 분석 (Hit@k & MRR)

이 노트북은 **HybridRetriever (RRF)**의 검색 성능, 즉 BM25와 Vector 검색이 결합되었을 때의 시너지를 정량적으로 평가합니다.

In [1]:
# ✅ 실험 파라미터 설정
SAMPLE_SIZE = 50
K_VALUES = [1, 3, 5]

# 환경 설정
import sys
import os
from dotenv import load_dotenv
import pandas as pd
import json
import random

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.append(project_root)

load_dotenv(os.path.join(project_root, ".env"))
print(f"Project Root: {project_root}")

Project Root: /Users/leemdo/Workspaces/SKN22-3rd-3Team


In [2]:
from src.retrieval.hybrid_search import HybridRetriever

retriever = HybridRetriever(version="v3", collection_name="care_guides")
print("✅ HybridRetriever initialized.")

✅ HybridRetriever initialized.


## 1. 데이터셋 로드

In [3]:
dataset_path = os.path.join(project_root, "data/v3/golden_dataset.json")

if os.path.exists(dataset_path):
    with open(dataset_path, "r", encoding="utf-8") as f:
        FULL_DATASET = json.load(f)
    print(f"📚 Loaded {len(FULL_DATASET)} test cases.")
else:
    print("⚠️ Dataset not found. Using mock.")
    FULL_DATASET = [{"query": "메인쿤 특징", "expected_keyword": "메인 쿤"}]

test_dataset = random.sample(FULL_DATASET, min(SAMPLE_SIZE, len(FULL_DATASET)))
print(f"🧪 Selected {len(test_dataset)} samples.")

📚 Loaded 3354 test cases.
🧪 Selected 50 samples.


## 2. 평가 로직 구현 (Hit@k, MRR)

In [4]:
async def evaluate_hybrid(dataset, k_values):
    results_summary = {k: 0 for k in k_values}
    total_mrr = 0
    log_data = []
    
    max_k = max(k_values)
    
    for item in dataset:
        query = item.get("query")
        target = item.get("expected_keyword", item.get("target"))
        
        if not target: continue
        
        specialist = item.get("specialist")
        search_results = await retriever.search(query, specialist=specialist, limit=max_k)
        
        found_rank = None
        for rank, res in enumerate(search_results):
            content = (
                res.get('name_ko', '') + " " + 
                res.get('title_refined', '') + " " + 
                res.get('text', '')
            ).lower()
            
            if target.lower() in content:
                found_rank = rank + 1
                break
        
        reciprocal_rank = 0
        hit_status = "❌"
        
        if found_rank:
            reciprocal_rank = 1 / found_rank
            hit_status = f"✅ (Rank {found_rank})"
            
            for k in k_values:
                if found_rank <= k:
                    results_summary[k] += 1
        
        total_mrr += reciprocal_rank
        
        log_data.append({
            "Query": query,
            "Target": target,
            "Result": hit_status,
            "MRR": round(reciprocal_rank, 4)
        })
        
    # 최종 집계
    total_count = len(dataset)
    metrics = {
        f"Hit@{k}": round(count / total_count, 4) 
        for k, count in results_summary.items()
    }
    metrics["MRR"] = round(total_mrr / total_count, 4)
    
    return metrics, pd.DataFrame(log_data)

print("Evaluator ready.")

Evaluator ready.


In [5]:
metrics, df_log = await evaluate_hybrid(test_dataset, K_VALUES)

print("="*40)
print("   📊 Hybrid (RRF) Evaluation Results")
print("="*40)
for k, v in metrics.items():
    print(f"{k:<10}: {v}")
print("="*40)

display(df_log.head(10))

🔍 [RETRIEVER]: '고양이를 스트레스 받지 않게 하려면?' 검색 중 (전문가: Liaison, 필터: None)...


Quantization is not supported for ArchType::neon. Fall back to non-quantized model.


📚 사용자 사전 로드 중: /Users/leemdo/Workspaces/SKN22-3rd-3Team/src/utils/../core/tokenizer/domain_dictionary.txt
🛑 28개의 불용어를 /Users/leemdo/Workspaces/SKN22-3rd-3Team/src/utils/../core/tokenizer/stopwords.txt에서 로드했습니다.
🔄 545개의 유의어 매핑을 /Users/leemdo/Workspaces/SKN22-3rd-3Team/src/utils/../core/tokenizer/synonyms.json에서 로드했습니다.
✅ [RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [RETRIEVER]: '고양이 관절 건강을 위한 팁은?' 검색 중 (전문가: Physician, 필터: None)...
✅ [RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [RETRIEVER]: '고양이가 편안함을 느끼는 사람의 특징은?' 검색 중 (전문가: Liaison, 필터: None)...
✅ [RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [RETRIEVER]: '피부병의 원인은 무엇인가요?' 검색 중 (전문가: Physician, 필터: None)...
✅ [RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [RETRIEVER]: '경계심 강한 고양이와 친해지는 방법은 무엇인가요?' 검색 중 (전문가: Peacekeeper, 필터: None)...
✅ [RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [RETRIEVER]: '고양이의 건강 상태를 어떻게 확인하나요?' 검색 중 (전문가: Physician, 필터: None)...
✅ [RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [RETRIEVER]: '10만 원 이하의 캣타워 추천 제품은?' 검색 중 (전문가: Liaison, 필터: None)...
✅ [RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [RETRIEVER]: '고양이에게 우유를 줘도 되

,Query,Target,Result,MRR
0,고양이를 스트레스 받지 않게 하려면?,고양이가 싫어하는 집사의 생활 습관 10가지,❌,0.0
1,고양이 관절 건강을 위한 팁은?,고양이 슬개골 탈구 및 관절 영양제 비교,❌,0.0
2,고양이가 편안함을 느끼는 사람의 특징은?,고양이가 편안함을 느끼는 집사의 특징 4가지,✅ (Rank 1),1.0
3,피부병의 원인은 무엇인가요?,고양이 피부병 증상 및 대처법,✅ (Rank 1),1.0
4,경계심 강한 고양이와 친해지는 방법은 무엇인가요?,경계심 강한 고양이와 친해지는 4가지 방법,✅ (Rank 1),1.0
5,고양이의 건강 상태를 어떻게 확인하나요?,고양이 앉은 자세와 편안함,❌,0.0
6,10만 원 이하의 캣타워 추천 제품은?,가성비 캣타워 및 캣폴 추천,✅ (Rank 1),1.0
7,고양이에게 우유를 줘도 되나요?,"고양이 우유, 안전한 제품 선택하기",✅ (Rank 1),1.0
8,뱅갈 고양이의 특징은 무엇인가요?,뱅갈 고양이 정보,✅ (Rank 1),1.0
9,고양이 차멀미 증상은 무엇인가요?,고양이 차멀미 증상 및 대처법,✅ (Rank 1),1.0
